# Creating Causal CelebA Dataset By Construction

We will walk through the construction of two causal CelebA datasets.

Gender <--> Age -> HairColor

Gender <--> Age -> Eyeglasses

These will be used in downstream experiments on disentangled causal representation learning

In [ ]:
%load_ext autoreload
%autoreload 2

In [3]:
from copy import copy
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torchvision.datasets import CelebA
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.utils import save_image, make_grid
import seaborn as sns
import matplotlib.pyplot as plt

from ciflows.datasets.causalceleba import CausalCelebA
from ciflows.datasets.multidistr import StratifiedSampler

from albumentations import (
    CoarseDropout,
    Compose,
    HorizontalFlip,
    OneOf,
    RandomCrop,
)
from albumentations.pytorch import ToTensorV2
from PIL import Image

In [12]:
# Root directory for the dataset
data_root = Path("/Users/adam2392/pytorch_data/")
data_root = Path('/local/eb/adam2392')
# Spatial size of training images, images are resized to this size.
image_size = 128

celeba_data = CelebA(
    data_root,
    download=True,
    target_type='identity',
    transform=transforms.Compose(
        [
            transforms.Resize(image_size),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ]
    ),
)

Files already downloaded and verified


## Haircolor Dataset

First, let's demonstrate the preprocessing needed to create the causal dataset involving hair color.

In [6]:
attr_names = celeba_data.attr_names
gender_idx = attr_names.index("Male")
age_idx = attr_names.index("Young")

blackhair_idx = attr_names.index("Black_Hair")
blondhair_idx = attr_names.index("Blond_Hair")
brownhair_idx = attr_names.index("Brown_Hair")
grayhair_idx = attr_names.index("Gray_Hair")
print(celeba_data.attr[:, gender_idx].shape)

print(
    blackhair_idx,
    blondhair_idx,
    brownhair_idx,
    grayhair_idx
)



torch.Size([162770])
8 9 11 17
['5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young', '']


In [18]:
# create a dataframe for the celebA attributes
df = pd.DataFrame(celeba_data.attr, columns=celeba_data.attr_names[:-1])
df['sample_idx'] = np.arange(len(df), dtype=int)

# now filter the dataframe based on meeting exactly one of the chosen hair colors
hair_colors = ['Black_Hair', 'Blond_Hair', 'Gray_Hair']
df_filtered = df[df[hair_colors].sum(axis=1) == 1]
df_filtered.reset_index(inplace=True, drop=True)

print(f'Number of total samples used: {len(df_filtered)} filtered from total of {len(df)} - {len(df_filtered) / len(df):.3f} of the total')
display(df_filtered.head())

Number of total samples used: 69193 filtered from total of 162770 - 0.425 of the total


,5_o_Clock_Shadow,Arched_Eyebrows,Attractive,Bags_Under_Eyes,Bald,Bangs,Big_Lips,Big_Nose,Black_Hair,Blond_Hair,...,Smiling,Straight_Hair,Wavy_Hair,Wearing_Earrings,Wearing_Hat,Wearing_Lipstick,Wearing_Necklace,Wearing_Necktie,Young,sample_idx
0,1,0,1,1,0,0,1,1,1,0,...,0,1,0,0,0,0,0,0,1,6
1,1,1,0,1,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,1,7
2,0,0,1,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,1,10
3,0,0,1,1,0,0,0,0,1,0,...,1,1,0,0,0,0,0,0,1,11
4,0,0,0,0,0,0,0,0,0,1,...,1,1,0,0,0,0,0,0,1,12
